## Load df's

hai, participants, flow, transcriptomics

In [1]:
from pathlib import Path

import pandas as pd

from data_cleaning.utils import peek

CLEANED_DIR = Path("cleaned_data")
RAW_DIR = Path("data")

In [2]:
hai_df = pd.read_csv(CLEANED_DIR / "hai_cleaned.csv", low_memory=False)
print(hai_df.shape)
peek(hai_df)

# Build the reference set of participant IDs — all other tables will be filtered to this
hai_participants = set(hai_df["participant_id"].unique())
print(f"Unique participants in HAI: {len(hai_participants)}")

(128177, 5)
Unique participants in HAI: 3775


In [3]:
unique_genes_challenge = set(
    pd.read_csv(RAW_DIR / "challenge_transcriptomics.tsv", sep='\t')['ensembl_gene_id'].unique()
)
len(unique_genes_challenge)

54902

In [4]:
participants_df = pd.read_csv(CLEANED_DIR / "participants_cleaned.csv", low_memory=False)
participants_df = participants_df[participants_df["participant_id"].isin(hai_participants)]
print(participants_df.shape)
peek(participants_df)

(3775, 6)


,participant_id,biological_sex,race,geolocation,arm_name,age
0,SDY269.SUB112836,Female,White,Georgia,Other,28.0
1,SDY269.SUB112849,Female,Black,Georgia,Other,39.0
2,SDY269.SUB112854,Male,Black,Georgia,Other,46.0
3,SDY269.SUB112860,Female,White,Georgia,Other,32.0
4,SDY269.SUB112881,Female,Black,Georgia,Other,29.0


In [5]:
flow_df = pd.read_csv(CLEANED_DIR / "flow_cleaned.csv", low_memory=False)
flow_df = flow_df[flow_df["participant_id"].isin(hai_participants)]
print(flow_df.shape)
peek(flow_df)

(136360, 8)


,participant_id,timepoint,parent_population,population_definition,name,unit,value,material
0,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+,Freq. of,WBC",live CD45+ cells,cells/uL,176.220000,blood
1,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ1: CD27,, IgD+,Freq. of,C...",Naive B cells,cells/uL,112.780800,blood
2,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ2: CD27+,, IgD+,Freq. of,...",Memory class switched + non-class switched B c...,cells/uL,12.370644,blood
3,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ3: CD27+,, IgD,Freq. of,C...",Memory class switched B cells,cells/uL,21.322620,blood
4,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ4: CD27,, IgD,Freq. of,CD19+",B cells,cells/uL,29.781180,blood


In [6]:
t2019 = pd.read_parquet(CLEANED_DIR / "transcriptomics_2019_UGA_cleaned.parquet")
t2020 = pd.read_parquet(CLEANED_DIR / "transcriptomics_2020_UGA_cleaned.parquet")
tsdy2867 = pd.read_parquet(CLEANED_DIR / "transcriptomics_SDY2867_cleaned.parquet")

# Merge on shared key columns so overlapping feature cols don't get duplicated
transcriptomics = (
    t2019
    .merge(t2020, on=["participant_id"], how="outer")
    .merge(tsdy2867, on=["participant_id"], how="outer")
)
transcriptomics = transcriptomics[transcriptomics["participant_id"].isin(hai_participants)]
print(transcriptomics.shape)

(396, 206637)


In [7]:
peek(transcriptomics)

,participant_id,d0_ENSG00000000003_x,d0_ENSG00000000005_x,d0_ENSG00000000419_x,d0_ENSG00000000457_x,d0_ENSG00000000460_x,d0_ENSG00000000938_x,d0_ENSG00000000971_x,d0_ENSG00000001036_x,d0_ENSG00000001084_x,d0_ENSG00000001167_x,d0_ENSG00000001460_x,d0_ENSG00000001461_x,d0_ENSG00000001497_x,d0_ENSG00000001561_x,d0_ENSG00000001617_x,d0_ENSG00000001626_x,d0_ENSG00000001629_x,d0_ENSG00000001630_x,d0_ENSG00000001631_x
0,2019_UGA.ID_001,1.001284,-0.123083,0.651376,0.546033,1.491505,-0.622255,-1.007577,0.598130,0.109457,0.388386,0.512635,1.018868,0.923870,0.007177,-0.310983,-0.184023,0.478017,0.767752,0.390241
1,2019_UGA.ID_005,0.586238,-0.123083,-0.682261,0.408613,0.482631,-0.929775,-0.492218,0.042291,0.819701,0.915067,0.428503,1.455299,1.194103,0.816484,0.350294,-0.184023,1.078850,0.598956,0.949021
2,2019_UGA.ID_008,0.268739,-0.123083,0.012367,0.200666,-0.070990,0.235228,-0.282735,0.089897,1.102819,-0.055020,1.049170,0.256959,0.117698,-0.487343,-0.310983,-0.184023,-0.535201,0.102128,-0.233563
3,2019_UGA.ID_011,-0.473472,-0.123083,-0.478352,0.620830,1.193616,0.432613,-0.732424,0.266911,1.158720,1.180399,-0.197465,0.385646,-0.057288,1.445018,-0.310983,-0.184023,1.007912,0.360122,0.999589
4,2019_UGA.ID_014,-1.145122,-0.123083,-0.007073,0.811627,0.804596,0.083290,-0.379761,0.929377,0.935906,1.007133,1.064422,1.308638,0.622721,1.428517,2.138805,-0.184023,0.931346,0.754686,0.775816


## Merge one by one

In [8]:
merged = participants_df.merge(flow_df, on="participant_id", how="left")
print(merged.shape)

(139994, 13)


In [9]:
merged = merged.merge(hai_df, on="participant_id", how="left", suffixes=("", "_hai"))
print(merged.shape)

(1285247, 17)


In [10]:
merged = merged.merge(transcriptomics, on="participant_id", how="left")
print(merged.shape)
peek(merged)

MemoryError: Unable to allocate 614. GiB for an array with shape (64124, 1285247) and data type float64